# 结构预测和处理
- /home/admin123/work/GTmining/diffpool/predict_data/

## 结构预测

In [ ]:
# python /home/admin123/esmfold/esm-main/scripts/fold.py -i domain_out.fasta -o structure -m /home/admin123/.cache/torch/hub/ --num-recycles 4

## 结构对齐

In [3]:
# ==========脚本内容：Protein_align.py==========
import subprocess
import os
from tqdm import tqdm

def get_all_files(directory):
    all_files = []
    # 遍历目录及其子目录
    for root, dirs, files in os.walk(directory):
        # 将每个文件的绝对路径添加到列表中
        for file in files:
            file_path = os.path.join(root, file)
            all_files.append(file_path)
    return all_files

def delete_files_in_folder(folder_path):
    # 获取文件夹下所有文件和子文件夹
    for root, dirs, files in os.walk(folder_path):
        # 删除所有文件
        for file in files:
            file_path = os.path.join(root, file)
            os.remove(file_path)

def delete_special_files_in_folder(folder_path, prefix, fila_n):
    # 获取文件夹下所有文件和子文件夹
    for root, dirs, files in os.walk(folder_path):
        # 删除所有文件
        for file in files:
            if file.endswith(prefix) and file.startswith(fila_n):
                file_path = os.path.join(root, file)
                os.remove(file_path)

fold_type = 'GTB'

directory = f"./predict_data/structure/"
storage_direcroey = f"./predict_data/structure_align/"
os.makedirs(storage_direcroey, exist_ok=True)
all_files = get_all_files(directory)
delete_files_in_folder(storage_direcroey)
# USalign结构比对==========
exe_path = './predict_data/exe/USalign'
cluster_center_pdb = f'./predict_data/cazy_cluster_center_{fold_type}.pdb'
# USalign AAA.pdb cluster.pdb -o AAA
for file in tqdm(all_files):
    file = file.split('/')[-1]
    temp1 = os.path.join(directory, file)
    temp2 = os.path.join(storage_direcroey, file.split('.pdb')[0])
    subprocess.run([exe_path, temp1, cluster_center_pdb, '-o', temp2], stdout=subprocess.DEVNULL)
    delete_special_files_in_folder(storage_direcroey, '.pml', file.split('.pdb')[0])


100%|██████████| 123/123 [00:25<00:00,  4.85it/s]


# 提取特征
- 切换为环境starG

## Step1 计算网格参数

In [4]:
# 提取局部结构特征第一步
import os

fold_type = 'GTB'

structure_path = './predict_data/structure_align/'
files = os.listdir(structure_path)

f = open('./predict_data/Protein_extract_feature_step1.sh', 'w')
for file in files:
    f.write(f"python ../../Predict_Extract_feature_step1.py {file} 6 6 {fold_type}\n")
f.close()

# nohup parallel --jobs 30 < Protein_extract_feature_step1.sh > Feature_Step_1.out 2>&1 &


## Step2 获取局部特征

In [6]:
# 提取局部结构特征第二步
import os

fold_type = 'GTB'

structure_path = './predict_data/structure_align/'
files = os.listdir(structure_path)

sample_redies_udp = 6
sample_redies_sugar = 6

step = 1
epoch = 1
f = open(f'./predict_data/Extract_feature_step2_epoch{epoch}.sh', 'w')
for file in files:
    if step % 40 == 0:
        f.write(f"taskset -c {step} python ../../Predict_Extract_feature_step2.py {file} {sample_redies_udp} {sample_redies_sugar} {fold_type}\n")
        f.close()
        epoch += 1
        f = open(f'./predict_data/Extract_feature_step2_epoch{epoch}.sh', 'w')
        step = 1
        continue
    f.write(f"taskset -c {step} python ../../Predict_Extract_feature_step2.py {file} {sample_redies_udp} {sample_redies_sugar} {fold_type}\n")
    step = step + 1
f.close()

f = open(f'./predict_data/Extract_feature_step2_AAArun.sh', 'w')
for e in range(epoch):
    f.write(f"parallel --jobs 40 < Extract_feature_step2_epoch{e+1}.sh\n")
f.close()

# nohup bash Extract_feature_step2_AAArun.sh &


# 生成数据集

In [7]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

fold_type = 'GTB'

sample_redies_udp = 6
sample_redies_sugar = 6

if fold_type == 'GTA':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6,
                        'dTDP-Rha': 7, 'Other': 8}
elif fold_type == 'GTB':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                        'dTDP-Rha': 8, 'Other': 9}
else:
    raise ValueError('Fold_type must be one of the GTA and GTB.')


# 获取有哪些local结构
storage_path = './predict_data/local_feature/'
dl_data_path = './predict_data/dl_data/'

local_list = [x.split('.npy')[0] for x in os.listdir(storage_path)]

# 生成文件列表
predict_process_path = []
for file in os.listdir(storage_path):
    npy_path = os.path.join(storage_path, file)
    predict_process_path.append(npy_path)

# 管理文件夹
if not os.path.isdir(dl_data_path):
    os.makedirs(dl_data_path, exist_ok=True)
else:
    shutil.rmtree(dl_data_path)
    os.makedirs(dl_data_path, exist_ok=True)
os.makedirs(f'{dl_data_path}/predict/')
os.makedirs(f'{dl_data_path}/predict/trace_file/')

# ==================== 生成数据 ====================
def make_database(process_path: list, data_type: str):
    f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
    f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
    f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
    f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
    f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
    edge_max = -1
    graph_idx = -1
    # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
    for file in tqdm(process_path):
        # +++++ 一次性读取npy文件 +++++
        try:
            input_dict = np.load(file, allow_pickle=True)
        except:
            print(f"wrong {file}")
            continue
        input_dict = input_dict[()]
        if len(input_dict['edges']) <= 100:
            # 用来检查不正确的局部网格
            print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
            continue
        # +++++ f_A的数据处理 +++++
        edges = input_dict['edges']
        edges += (edge_max +1)
        for edge in edges:
            f_A.write(f"{edge[0]}, {edge[1]}\n")
        edge_max = np.max(edges)
        # +++++ f_graph_indicator的数据处理 +++++
        graph_idx += 1
        xyzs = input_dict['xyz']
        for i in range(0, xyzs.shape[0]):
            f_graph_indicator.write(f"{graph_idx}\n")
        # +++++ f_graph_labels的数据处理 +++++
        f_graph_labels.write("0\n")
        # +++++ f_node_attributes的数据处理 +++++
        xyzs = input_dict['xyz']
        sis = input_dict['si']
        hbonds = input_dict['hbond']
        charges = input_dict['charge']
        hphobs = input_dict['hphob']
        trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
        f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
        for i in range(0, xyzs.shape[0]):
            # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
            # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
            # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
            x = xyzs[i][0]
            y = xyzs[i][1]
            z = xyzs[i][2]
            f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
            f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
        f_trace_file.close()
        f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
        temp_correspond_information = ''
        temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                            round(xyzs[0][1], 5),
                                                                                                            round(xyzs[0][2], 5),
                                                                                                            round(sis[0][0], 5),
                                                                                                            round(hbonds[0][0], 5),
                                                                                                            round(charges[0][0], 5),
                                                                                                            round(hphobs[0][0], 5))
        temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                round(xyzs[1][1], 5),
                                                                                                                round(xyzs[1][2], 5),
                                                                                                                round(sis[1][0], 5),
                                                                                                                round(hbonds[1][0], 5),
                                                                                                                round(charges[1][0], 5),
                                                                                                                round(hphobs[1][0], 5))
        temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                round(xyzs[2][1], 5),
                                                                                                                round(xyzs[2][2], 5),
                                                                                                                round(sis[2][0], 5),
                                                                                                                round(hbonds[2][0], 5),
                                                                                                                round(charges[2][0], 5),
                                                                                                                round(hphobs[2][0], 5))
        f_correspond_information.write(temp_correspond_information)

    f_A.close()
    f_graph_indicator.close()
    f_graph_labels.close()
    f_node_attributes.close()
    f_correspond_information.close()

make_database(predict_process_path, 'predict')

100%|██████████| 123/123 [00:00<00:00, 154.61it/s]


## 检查数据集会不会报错
- 切换环境为dgl

In [1]:
import argparse
from tqdm import tqdm
import dgl
import torch
import torch.nn.functional as F
from dgl.data import tu
from model.encoder import DiffPool

def prepare_data(dataset, shuffle=False, prog_args=None):
    """
    preprocess TU dataset according to DiffPool's paper setting and load dataset into dataloader
    """
    return dgl.dataloading.GraphDataLoader(
        dataset,
        batch_size=prog_args.batch_size,
        shuffle=shuffle,
        num_workers=prog_args.n_worker,
    )


def train(dataset, model, prog_args, id_card_protein):
    """
    training function
    """
    dataloader = dataset

    if prog_args.cuda > 0:
        torch.cuda.set_device(0)
    
    for epoch in range(prog_args.epoch):
        model.train()
        print("\nEPOCH ###### {} ######".format(epoch))
        for batch_idx, (batch_graph, graph_labels) in tqdm(enumerate(dataloader), total=len(dataloader)):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            model.zero_grad()
            # ypred = model(batch_graph)
            temp = batch_graph
            protein_temp_id_card = ''
            protein_temp_id_card = protein_temp_id_card + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][0][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][1][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][2][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][6]), 5))
            try:
                ypred = model(batch_graph)
            except:
                print('This is the protein name not be forward ==========', id_card_protein[protein_temp_id_card])
                continue
            try:
                ttt = id_card_protein[protein_temp_id_card]
            except:
                print('This is the data not be traced ==========', protein_temp_id_card)
        torch.cuda.empty_cache()
        break
    return 'Trian successfully'


/home/admin123/software/anaconda3/envs/dgl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sample_redies_udp = 6
sample_redies_sugar = 6


global_train_time_per_epoch = []

print("{:=^100}".format('prog_args'))
global_train_time_per_epoch = []
prog_args = argparse.Namespace(dataset=f'GTmining_6_6', pool_ratio=0.10, num_pool=1, cuda=1, lr=1e-4, clip=1.0,
                               batch_size=1, epoch=1, train_ratio=0.7, test_ratio=0.1, n_worker=2, gc_per_block=3,
                               dropout=0.20, method="diffpool", bn=True, bias=True, save_dir="./model_param",
                               load_epoch=-1, data_mode="default", linkpred=False, hidden_dim=64, embedding_dim=64)
print(prog_args)

print("{:=^100}".format('加载数据'))
dataset_train = tu.RhaFinderDataset(name="GTmining",
                                    raw_dir=f'../data/dl_data/GTA/fold1/train/')
dataset_validation = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTA/fold1/validation/')
dataset_test = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTA/fold1/test/')
dataset_nova = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTA/fold1/nova/')
train_dataloader = prepare_data(dataset_train, shuffle=True, prog_args=prog_args)
validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)
test_dataloader = prepare_data(dataset_test, shuffle=False, prog_args=prog_args)
nova_dataloader = prepare_data(dataset_nova, shuffle=False, prog_args=prog_args)

dataset_predict = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'./predict_data/dl_data/predict/')
predict_dataloader = prepare_data(dataset_predict, shuffle=False, prog_args=prog_args)

input_dim_train, label_dim_train, max_num_node_train = dataset_train.statistics()
input_dim_validation, label_dim_validation, max_num_node_validation = dataset_validation.statistics()
input_dim_test, label_dim_test, max_num_node_test = dataset_test.statistics()
input_dim_nova, label_dim_nova, max_num_node_nova = dataset_nova.statistics()
max_num_node = max([max_num_node_train, max_num_node_validation, max_num_node_test, max_num_node_nova])
input_dim = input_dim_train
label_dim = label_dim_train
print("++++++++++ STATISTICS ABOUT THE DATASET ++++++++++")
print("dataset feature dimension is", input_dim_train)
print("dataset label dimension is", label_dim_train)
print("the max num node is", max_num_node)
print("number of graphs is", len(dataset_train) + len(dataset_validation)+ len(dataset_test)+ len(dataset_nova))

hidden_dim = prog_args.hidden_dim  # used to be 64
embedding_dim = prog_args.embedding_dim

assign_dim = int(max_num_node * prog_args.pool_ratio)
print("++++++++++MODEL STATISTICS++++++++")
print("model hidden dim is", hidden_dim)
print("model embedding dim for graph instance embedding", embedding_dim)
print("initial batched pool graph dim is", assign_dim)
activation = F.relu

# initialize model
model = DiffPool(
    input_dim,
    hidden_dim,
    embedding_dim,
    label_dim,
    activation,
    prog_args.gc_per_block,
    prog_args.dropout,
    prog_args.num_pool,
    prog_args.linkpred,
    prog_args.batch_size,
    "meanpool",
    assign_dim,
    prog_args.pool_ratio,
    []
)

print("model init finished")
print("MODEL:::::::", prog_args.method)
if prog_args.cuda:
    model = model.cuda()

=============================================prog_args==============================================
Namespace(dataset='GTmining_6_6', pool_ratio=0.1, num_pool=1, cuda=1, lr=0.0001, clip=1.0, batch_size=1, epoch=1, train_ratio=0.7, test_ratio=0.1, n_worker=2, gc_per_block=3, dropout=0.2, method='diffpool', bn=True, bias=True, save_dir='./model_param', load_epoch=-1, data_mode='default', linkpred=False, hidden_dim=64, embedding_dim=64)
================================================加载数据================================================
++++++++++ STATISTICS ABOUT THE DATASET ++++++++++
dataset feature dimension is 7
dataset label dimension is 9
the max num node is 537
number of graphs is 11489
++++++++++MODEL STATISTICS++++++++
model hidden dim is 64
model embedding dim for graph instance embedding 64
initial batched pool graph dim is 53
model init finished
MODEL::::::: diffpool


In [3]:
# train_dataloader  validation_dataloader  test_dataloader  nova_dataloader  predict_dataloader
# train  validation  test  nova  predict

id_card_protein = {}
with open(f'./predict_data/dl_data/predict/Predict_correspond_information.txt', 'r')as f:
    for dd in f.readlines():
        dd = dd.split('\n')[0].split('===')
        id_card_protein[dd[1]] = dd[0]

logger = train(
    predict_dataloader, model, prog_args, id_card_protein
)

print("Ending!!!")


EPOCH ###### 0 ######


100%|██████████| 123/123 [00:06<00:00, 18.31it/s]

Ending!!!


# 模型预测

In [1]:
fold_num = 1
family_fold_type = 'GTB'

## 函数定义

In [2]:
import argparse
import textwrap
import os
import time
import dgl
import torch
import torch.nn as nn
import torch.nn.functional as F
from dgl.data import tu
from model.encoder import DiffPool
from livelossplot import PlotLosses
from sklearn.metrics import f1_score
import shutil
import pandas as pd

def prepare_data(dataset, shuffle=False, prog_args=None, custom_batch_size=None):
    """
    preprocess TU dataset according to DiffPool's paper setting and load dataset into dataloader
    """
    if custom_batch_size is None:
        return dgl.dataloading.GraphDataLoader(
            dataset,
            batch_size=prog_args.batch_size,
            shuffle=shuffle,
            num_workers=prog_args.n_worker,
        )
    else:
        return dgl.dataloading.GraphDataLoader(
            dataset,
            batch_size=custom_batch_size,
            shuffle=shuffle,
            num_workers=prog_args.n_worker,
        )


print("{:=^100}".format(f'fold num is : {fold_num}, family type is : {family_fold_type}'))

print("{:=^100}".format('prog_args'))
prog_args = argparse.Namespace(dataset=f'GTmining_6_6_{family_fold_type}_fold{fold_num}', pool_ratio=0.10, num_pool=1, cuda=1, lr=1.0, clip=float("inf"),
                               batch_size=128, epoch=500, n_worker=10, gc_per_block=3, aggregator_type="meanpool",
                               dropout=0.00, method="diffpool", bn=True, bias=True, save_dir=f"./model_param",
                               load_epoch=-1, data_mode="default", linkpred=False, hidden_dim=64, embedding_dim=64, family_fold_type=family_fold_type)
print( textwrap.fill(str(prog_args), width=100))

print("{:=^100}".format('加载数据'))
dataset_train = tu.RhaFinderDataset(name="GTmining",
                                    raw_dir=f'../data/dl_data/{family_fold_type}/fold{fold_num}/train/')
dataset_validation = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}/fold{fold_num}/validation/')
dataset_test = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}/fold{fold_num}/test/')
dataset_nova = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}/fold{fold_num}/nova/')
train_dataloader = prepare_data(dataset_train, shuffle=True, prog_args=prog_args)
validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)
test_dataloader = prepare_data(dataset_test, shuffle=False, prog_args=prog_args)
nova_dataloader = prepare_data(dataset_nova, shuffle=False, prog_args=prog_args)

dataset_predict = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'./predict_data/dl_data/predict/')
predict_dataloader = prepare_data(dataset_predict, shuffle=False, prog_args=prog_args, custom_batch_size=1)


input_dim_train, label_dim_train, max_num_node_train = dataset_train.statistics()
input_dim_validation, label_dim_validation, max_num_node_validation = dataset_validation.statistics()
input_dim_test, label_dim_test, max_num_node_test = dataset_test.statistics()
input_dim_nova, label_dim_nova, max_num_node_nova = dataset_nova.statistics()
max_num_node = max([max_num_node_train, max_num_node_validation, max_num_node_test, max_num_node_nova])
input_dim = input_dim_train
label_dim = label_dim_train
print("++++++++++ STATISTICS ABOUT THE DATASET ++++++++++")
print("dataset feature dimension is", input_dim_train)
print("dataset label dimension is", label_dim_train)
print("the max num node is", max_num_node)
print("number of graphs is", len(dataset_train) + len(dataset_validation)+ len(dataset_test)+ len(dataset_nova))

hidden_dim = prog_args.hidden_dim  # used to be 64
embedding_dim = prog_args.embedding_dim

assign_dim = int(max_num_node * prog_args.pool_ratio)
print("++++++++++MODEL STATISTICS++++++++")
print("model hidden dim is", hidden_dim)
print("model embedding dim for graph instance embedding", embedding_dim)
print("initial batched pool graph dim is", assign_dim)
activation = F.relu

if family_fold_type == 'GTA':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6,
                        'dTDP-Rha': 7, 'Other': 8}
elif family_fold_type == 'GTB':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                        'dTDP-Rha': 8, 'Other': 9}
else:
    raise ValueError(f"Invalid family_fold_type: '{prog_args.family_fold_type}'. Valid options are 'GTA' and 'GTB'.")
df_cluster = pd.read_excel(f'../data/cluster/{family_fold_type}/dataseat_split_{fold_num}.xlsx')
df_cluster = df_cluster.loc[df_cluster['Dataset']=='train']
df_cluster.reset_index(drop=True, inplace=True)
custom_loss_weight = []
total_sample = df_cluster.shape[0]
for x in graph_label_dict.keys():
    df_x = df_cluster.loc[df_cluster['Activate']==x]
    df_x.reset_index(drop=True, inplace=True)
    x_sample = df_x.shape[0]
    custom_loss_weight.append(total_sample/x_sample)

assert len(custom_loss_weight) == label_dim_train, 'Wrong custom loss weight, please check what happen.'

# initialize model
model = DiffPool(
    input_dim,
    hidden_dim,
    embedding_dim,
    label_dim,
    activation,
    prog_args.gc_per_block,
    prog_args.dropout,
    prog_args.num_pool,
    prog_args.linkpred,
    prog_args.batch_size,
    prog_args.aggregator_type,
    assign_dim,
    prog_args.pool_ratio,
    custom_loss_weight
)
print("model init finished")
print("MODEL:::::::", prog_args.method)
if prog_args.cuda:
    model = model.cuda()




/home/admin123/software/anaconda3/envs/dgl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


===============================fold num is : 1, family type is : GTB================================
=============================================prog_args==============================================
Namespace(dataset='GTmining_6_6_GTB_fold1', pool_ratio=0.1, num_pool=1, cuda=1, lr=1.0, clip=inf,
batch_size=128, epoch=500, n_worker=10, gc_per_block=3, aggregator_type='meanpool', dropout=0.0,
method='diffpool', bn=True, bias=True, save_dir='./model_param', load_epoch=-1, data_mode='default',
linkpred=False, hidden_dim=64, embedding_dim=64, family_fold_type='GTB')
================================================加载数据================================================
++++++++++ STATISTICS ABOUT THE DATASET ++++++++++
dataset feature dimension is 7
dataset label dimension is 10
the max num node is 655
number of graphs is 19896
++++++++++MODEL STATISTICS++++++++
model hidden dim is 64
model embedding dim for graph instance embedding 64
initial batched pool graph dim is 65
model init finished

In [3]:
# 先过一遍train_dataloader，让模型中的一些参数先初始化一下
model.train()
for batch_idx, (batch_graph, graph_labels) in enumerate(train_dataloader):
    for key, value in batch_graph.ndata.items():
        batch_graph.ndata[key] = value.float()
    graph_labels = graph_labels.long()
    if torch.cuda.is_available():
        batch_graph = batch_graph.to(torch.cuda.current_device())
        graph_labels = graph_labels.cuda()
    ypred = model(batch_graph)
    loss = model.loss(ypred, graph_labels)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=prog_args.clip)
    model.zero_grad()



## 开始预测

In [11]:
import pandas as pd

GTA_best_epoch = {}
for x in range(1, 11):
    df_path = os.path.join(prog_args.save_dir, f'GTmining_6_6_GTA_fold{x}', 'validation_log.csv')
    df = pd.read_csv(df_path)
    best_epoch = df.loc[df['validation_f1_score'].idxmax(), 'epoch']
    GTA_best_epoch[x] = int(best_epoch)

GTB_best_epoch = {}
for x in range(1, 11):
    df_path = os.path.join(prog_args.save_dir, f'GTmining_6_6_GTB_fold{x}', 'validation_log.csv')
    df = pd.read_csv(df_path)
    best_epoch = df.loc[df['validation_f1_score'].idxmax(), 'epoch']
    GTB_best_epoch[x] = int(best_epoch)
    xx = df['validation_f1_score'][best_epoch]
    xxx = df['test_f1_score'][best_epoch]
    xxxx = df['nova_f1_score'][best_epoch]
    print(f'Best epoch for GTB fold {x} is {best_epoch}, validation f1 score is {xx}, test f1 score is {xxx}, nova f1 score is {xxxx}.')

id_card_protein = {}
with open(f'./predict_data/dl_data/predict/Predict_correspond_information.txt', 'r')as f:
    for dd in f.readlines():
        dd = dd.split('\n')[0].split('===')
        id_card_protein[dd[1]] = dd[0]

print(GTA_best_epoch)
print(GTB_best_epoch)



Best epoch for GTB fold 1 is 316, validation f1 score is 0.8234489104235339, test f1 score is 0.728960034172425, nova f1 score is 0.1160439761979334.
Best epoch for GTB fold 2 is 447, validation f1 score is 0.7628767893837057, test f1 score is 0.7657175823345544, nova f1 score is 0.0667500577007975.
Best epoch for GTB fold 3 is 419, validation f1 score is 0.785752892022684, test f1 score is 0.8364906139159123, nova f1 score is 0.0942394216163972.
Best epoch for GTB fold 4 is 491, validation f1 score is 0.8726463205008012, test f1 score is 0.659521201330241, nova f1 score is 0.0911931159152793.
Best epoch for GTB fold 5 is 499, validation f1 score is 0.8371655750671397, test f1 score is 0.7027207428760833, nova f1 score is 0.1712017311954953.
Best epoch for GTB fold 6 is 338, validation f1 score is 0.7856974241406034, test f1 score is 0.7925952564452078, nova f1 score is 0.1625316793881905.
Best epoch for GTB fold 7 is 479, validation f1 score is 0.7384276297221521, test f1 score is 0.7

In [12]:
df

,epoch,validation_accuracy,validation_f1_score,validation_f1_score_class_0,validation_f1_score_class_1,validation_f1_score_class_2,validation_f1_score_class_3,validation_f1_score_class_4,validation_f1_score_class_5,validation_f1_score_class_6,...,nova_f1_score_class_0,nova_f1_score_class_1,nova_f1_score_class_2,nova_f1_score_class_3,nova_f1_score_class_4,nova_f1_score_class_5,nova_f1_score_class_6,nova_f1_score_class_7,nova_f1_score_class_8,nova_f1_score_class_9
0,0,1.554252,0.003061,0.002801,0.004024,0.047619,0.040816,0.031169,0.133333,0.006897,...,0.001311,0.222222,0.007605,0.090909,0.000911,0.400000,0.009804,0.028169,0.029851,0.043478
1,1,14.516129,0.025352,0.002801,0.253903,0.047619,0.040816,0.036364,0.133333,0.006897,...,0.001311,0.007263,0.007605,0.090909,1.000000,0.400000,0.009804,0.028169,0.029851,0.043478
2,2,1.173021,0.002319,0.002801,0.004024,0.023754,0.040816,0.036364,0.133333,0.006897,...,0.001311,0.222222,0.213268,0.090909,1.000000,0.400000,0.009804,0.028169,0.029851,0.043478
3,3,1.173021,0.002319,0.002801,0.004024,0.023754,0.040816,0.036364,0.133333,0.006897,...,0.001311,0.222222,0.213703,0.090909,1.000000,0.200000,0.009804,0.028169,0.029851,0.043478
4,4,30.733138,0.090772,0.562574,0.004024,0.047619,0.040816,0.036364,0.133333,0.006897,...,0.326590,0.222222,0.007605,0.090909,1.000000,0.400000,0.009804,0.081680,0.029851,0.043478
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,78.973607,0.651762,0.781287,0.486486,0.926829,0.484076,0.036364,0.848485,0.102804,...,0.020685,0.009259,0.184818,0.027027,1.000000,0.142857,0.009804,0.285714,0.026667,0.032815
496,496,83.343109,0.729334,0.915900,0.486420,0.917647,0.722892,0.031250,0.933333,0.369958,...,0.135650,0.018692,0.183673,0.036810,0.500000,0.333333,0.009804,0.279476,0.011050,0.027027
497,497,81.026393,0.686484,0.861670,0.486692,0.950000,0.645833,0.035714,0.812500,0.183007,...,0.110723,0.009479,0.030075,0.025135,0.500000,0.333333,0.009804,0.320000,0.076923,0.026882
498,498,82.697947,0.719003,0.947228,0.473301,0.864865,0.800000,0.035714,0.896552,0.277311,...,0.055485,0.009434,0.007605,0.026718,0.500000,0.333333,0.009756,0.277372,0.024390,0.027157


In [13]:
from collections import defaultdict, Counter

if family_fold_type == 'GTA':
    best_epoch_dict = GTA_best_epoch
elif family_fold_type == 'GTB':
    best_epoch_dict = GTB_best_epoch
else:
    raise ValueError('SSSSSB')

predict_result = defaultdict(list)

for fold_num_predict in range(1, len(list(best_epoch_dict.keys()))+1):
    epoch_predict = best_epoch_dict[fold_num_predict]

    begin_time = time.time()
    print("\nEPOCH ###### Fold {} Epoch {} ######".format(fold_num_predict, epoch_predict))
    if epoch_predict is not None and prog_args.save_dir is not None:
        model.load_state_dict(
            torch.load(
                prog_args.save_dir
                + "/"
                + f'GTmining_6_6_{family_fold_type}_fold{fold_num_predict}'
                + "/model.iter-"
                + "{:04d}".format(epoch_predict), weights_only=True
            )
        )


    model.eval()
    with torch.no_grad():
        val_pred_indi = torch.tensor([], device='cuda')
        val_label_indi = torch.tensor([], device='cuda')
        for batch_idx, (batch_graph, graph_labels) in enumerate(predict_dataloader):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            # 拿标签
            temp = batch_graph
            protein_temp_id_card = ''
            protein_temp_id_card = protein_temp_id_card + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][0][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][1][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][2][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][6]), 5))
            protein_id = id_card_protein[protein_temp_id_card]


            ypred = model(batch_graph)
            indi = torch.argmax(ypred, dim=1)
            predict_result[protein_id].append(int(indi.cpu()))
            # print(f"Protein name is : {protein_id}, and its activate is {int(indi.cpu())}.")
            
    elapsed_time = time.time() - begin_time
    print("epoch {:.4f} with epoch time {:.4f} s".format(epoch_predict, elapsed_time))


EPOCH ###### Fold 1 Epoch 316 ######
epoch 316.0000 with epoch time 5.4685 s

EPOCH ###### Fold 2 Epoch 447 ######
epoch 447.0000 with epoch time 5.5394 s

EPOCH ###### Fold 3 Epoch 419 ######
epoch 419.0000 with epoch time 5.5366 s

EPOCH ###### Fold 4 Epoch 491 ######
epoch 491.0000 with epoch time 5.7900 s

EPOCH ###### Fold 5 Epoch 499 ######
epoch 499.0000 with epoch time 5.3805 s

EPOCH ###### Fold 6 Epoch 338 ######
epoch 338.0000 with epoch time 5.4835 s

EPOCH ###### Fold 7 Epoch 479 ######
epoch 479.0000 with epoch time 5.4790 s

EPOCH ###### Fold 8 Epoch 341 ######
epoch 341.0000 with epoch time 5.6029 s

EPOCH ###### Fold 9 Epoch 362 ######
epoch 362.0000 with epoch time 5.5009 s

EPOCH ###### Fold 10 Epoch 329 ######
epoch 329.0000 with epoch time 5.5554 s


In [14]:
# 存储结果：key为原键，value为(元素, 出现次数)的列表
result = {}

for key, values in predict_result.items():
    # 统计每个元素的出现次数
    count = Counter(values)
    
    if not count:  # 处理空列表情况（示例中没有空列表，仅作兼容）
        result[key] = []
        continue
    
    # 找到最大出现次数
    max_freq = max(count.values())
    
    # 筛选出所有出现次数等于最大次数的元素
    most_common = [(num, freq) for num, freq in count.items() if freq == max_freq]
    
    result[key] = most_common

# 打印结果
for key, items in result.items():
    print(f"{key}:")
    print(predict_result[key])
    for num, freq in items:
        print(f"  元素 {num}，出现次数 {freq}")
    print()  # 空行分隔

WP_080390664.1_d:
[1, 0, 9, 0, 3, 2, 0, 6, 0, 9]
  元素 0，出现次数 4

WP_077423872.1_d:
[0, 0, 9, 0, 0, 0, 9, 0, 0, 7]
  元素 0，出现次数 7

Rl_d:
[1, 1, 9, 9, 1, 9, 1, 1, 1, 1]
  元素 1，出现次数 7

Cc3_d:
[3, 1, 7, 8, 1, 7, 1, 1, 7, 7]
  元素 1，出现次数 4
  元素 7，出现次数 4

AYR23380.1_d:
[3, 0, 9, 9, 0, 0, 0, 9, 0, 9]
  元素 0，出现次数 5

Rp_d:
[3, 3, 6, 3, 1, 2, 7, 0, 1, 7]
  元素 3，出现次数 3

ALP62319.1_d:
[0, 3, 0, 7, 3, 2, 0, 2, 0, 9]
  元素 0，出现次数 4

WP_005821204.1_d:
[3, 2, 2, 0, 0, 2, 0, 9, 0, 9]
  元素 0，出现次数 4

WP_051977456.1_d:
[1, 3, 9, 3, 3, 0, 9, 0, 1, 9]
  元素 3，出现次数 3
  元素 9，出现次数 3

WP_180978721.1_d:
[0, 6, 0, 0, 9, 0, 0, 0, 0, 0]
  元素 0，出现次数 8

WP_006248446.1_d:
[0, 6, 0, 0, 9, 0, 9, 0, 7, 9]
  元素 0，出现次数 5

Pc_d:
[1, 2, 0, 7, 1, 9, 0, 0, 0, 9]
  元素 0，出现次数 4

Pp_d:
[1, 2, 6, 0, 3, 9, 9, 9, 2, 1]
  元素 9，出现次数 3

ANF68130.1_d:
[3, 3, 0, 7, 1, 2, 9, 9, 0, 9]
  元素 9，出现次数 3

Rsp2_d:
[3, 1, 9, 7, 1, 1, 3, 0, 9, 3]
  元素 3，出现次数 3
  元素 1，出现次数 3

VEI75114.1_d:
[0, 0, 9, 9, 0, 0, 0, 0, 0, 9]
  元素 0，出现次数 7

Cc1_d:
[9, 9, 9, 9,